In [2]:
# ==============================================================================
# Master Colab Evaluation: SciBERT, RoBERTa, Gemma 2 2B & Mistral 7B
# ==============================================================================
import os
import re
import csv
import json
import torch
import urllib.request
import numpy as np
from google.colab import drive
from tqdm.auto import tqdm

# --- TOGGLES: Turn ON/OFF any model you want to run ---
RUN_SCIBERT  = True   # ~15 seconds on T4 GPU
RUN_ROBERTA  = True   # ~15 seconds on T4 GPU
RUN_GEMMA    = True   # ~10 minutes on T4 GPU (Gemma 2 2B IT Verification)
RUN_MISTRAL  = True   # ~15 minutes on T4 GPU (Mistral 7B Instruct v0.3)

# 1. Mount Google Drive
drive.mount('/content/drive')

OUTPUT_DIR = "/content/drive/MyDrive/NLP_Checkpoints/error_analysis_empirical"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. Authoritative Banking77 77-Class Taxonomy Order
BANKING77_LABELS = [
    "activate_my_card", "age_limit", "apple_pay_or_google_pay", "atm_support",
    "automatic_top_up", "balance_not_updated_after_bank_transfer",
    "balance_not_updated_after_cheque_or_cash_deposit", "beneficiary_not_allowed",
    "cancel_transfer", "card_about_to_expire", "card_acceptance", "card_arrival",
    "card_delivery_estimate", "card_linking", "card_not_working",
    "card_payment_fee_charged", "card_payment_not_recognised",
    "card_payment_wrong_exchange_rate", "card_swallowed", "cash_withdrawal_charge",
    "cash_withdrawal_not_recognised", "change_pin", "compromised_card",
    "contactless_not_working", "country_support", "declined_card_payment",
    "declined_cash_withdrawal", "declined_transfer",
    "direct_debit_payment_not_recognised", "disposable_card_limits",
    "edit_personal_details", "exchange_charge", "exchange_rate", "exchange_via_app",
    "extra_charge_on_statement", "failed_transfer", "fiat_currency_support",
    "get_disposable_virtual_card", "get_physical_card", "getting_spare_card",
    "getting_virtual_card", "lost_or_stolen_card", "lost_or_stolen_phone",
    "order_physical_card", "passcode_forgotten", "pending_card_payment",
    "pending_cash_withdrawal", "pending_top_up", "pending_transfer", "pin_blocked",
    "receiving_money", "Refund_not_showing_up", "request_refund",
    "reverted_card_payment?", "supported_cards_and_currencies", "terminate_account",
    "top_up_by_bank_transfer_charge", "top_up_by_card_charge",
    "top_up_by_cash_or_cheque", "top_up_failed", "top_up_limits", "top_up_reverted",
    "topping_up_by_card", "transaction_charged_twice", "transfer_fee_charged",
    "transfer_into_account", "transfer_not_received_by_recipient", "transfer_timing",
    "unable_to_verify_identity", "verify_my_identity", "verify_source_of_funds",
    "verify_top_up", "virtual_card_not_working", "visa_or_mastercard",
    "why_verify_identity", "wrong_amount_of_cash_received",
    "wrong_exchange_rate_for_cash_withdrawal"
]

# 3. Download Canonical Banking77 Test Set from Official PolyAI Repository
TEST_CSV_PATH = "test.csv"
if not os.path.exists(TEST_CSV_PATH):
    print("--> Downloading canonical 3,080 test queries from official PolyAI repo...")
    url = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/test.csv"
    urllib.request.urlretrieve(url, TEST_CSV_PATH)

test_texts, test_labels = [], []
with open(TEST_CSV_PATH, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        test_texts.append(row["text"])
        test_labels.append(row["category"])

print(f"--> [READY] Loaded {len(test_texts):,} test queries across {len(BANKING77_LABELS)} classes.")

# ------------------------------------------------------------------------------
# PART 1: Discriminative BERT Encoders (SciBERT & RoBERTa)
# ------------------------------------------------------------------------------
from transformers import AutoTokenizer, AutoModelForSequenceClassification

BERT_ROOT = "/content/drive/MyDrive/NLP_Checkpoints/BERT_training_Awais_Seed42"
encoders = []

if RUN_SCIBERT:
    encoders.append({"name": "SciBERT", "path": os.path.join(BERT_ROOT, "SciBERT")})
if RUN_ROBERTA:
    # Check both "RoBERTa" and "RoBERTa-Base"
    r_path = os.path.join(BERT_ROOT, "RoBERTa") if os.path.exists(os.path.join(BERT_ROOT, "RoBERTa")) else os.path.join(BERT_ROOT, "RoBERTa-Base")
    encoders.append({"name": "RoBERTa", "path": r_path})

for enc in encoders:
    e_name = enc["name"]
    e_path = enc["path"]
    if not os.path.exists(e_path):
        print(f"[WARNING] Checkpoint path not found: {e_path}. Skipping.")
        continue

    print("\n" + "="*70)
    print(f"--> EVALUATING {e_name} ON 3,080 TEST QUERIES...")
    print("="*70)

    tokenizer = AutoTokenizer.from_pretrained(e_path)
    model = AutoModelForSequenceClassification.from_pretrained(e_path).to("cuda")
    model.eval()

    preds = []
    with torch.no_grad():
        for i in tqdm(range(0, len(test_texts), 32), desc=f"Evaluating {e_name}"):
            batch = test_texts[i : i + 32]
            inputs = tokenizer(batch, padding=True, truncation=True, max_length=64, return_tensors="pt").to("cuda")
            logits = model(**inputs).logits
            p_indices = torch.argmax(logits, dim=-1).cpu().numpy()
            # Map index directly to canonical taxonomy label string!
            preds.extend([BANKING77_LABELS[idx] for idx in p_indices])

    acc = (np.array(preds) == np.array(test_labels)).mean() * 100.0
    print(f">>> [CONFIRMED] {e_name} Test Accuracy: {acc:.2f}% <<<")

    out_file = os.path.join(OUTPUT_DIR, f"{e_name.lower()}_seed42_test_predictions.json")
    with open(out_file, "w", encoding="utf-8") as f:
        json.dump({"y_true": test_labels, "y_pred": preds, "texts": test_texts}, f)
    print(f"--> [SAVED] {out_file}")

    del model, tokenizer
    torch.cuda.empty_cache()

# ------------------------------------------------------------------------------
# PART 2: Generative QLoRA Models (Gemma 2 2B & Mistral 7B)
# ------------------------------------------------------------------------------
if RUN_GEMMA or RUN_MISTRAL:
    print("\n--> Installing Unsloth & dependencies...")
    !pip install unsloth unsloth_zoo
    !pip install --no-deps trl peft accelerate bitsandbytes

    os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
    torch._dynamo.config.suppress_errors = True
    from unsloth import FastLanguageModel

    few_shot_pool = [
        {"text": "I am still waiting on my new debit card to arrive after two weeks.", "label_name": "card_arrival"},
        {"text": "Why was an international transfer fee deducted from my account balance?", "label_name": "transfer_fee_charged"},
        {"text": "I lost my physical card yesterday and suspect someone is using it.", "label_name": "lost_or_stolen_card"}
    ]

    def create_few_shot_prompt(query):
        p = "You are an expert banking customer service AI assistant. Below are example customer queries and their correct intent categories:\n\n"
        for idx, ex in enumerate(few_shot_pool, 1):
            p += f"### Example {idx}:\nQuery: \"\"\"\n{ex['text'].strip()}\n\"\"\"\nIntent: {ex['label_name']}\n\n"
        p += f"### Target Query to Classify:\nQuery: \"\"\"\n{query.strip()}\n\"\"\"\nProvide ONLY the exact intent category name as your final output.\nIntent:"
        return p

    def parse_pred(gen_text):
        clean = gen_text.strip().strip('"').strip("'").strip().split("\n")[0].strip()
        for l in BANKING77_LABELS:
            if clean.lower() == l.lower():
                return l
        for l in BANKING77_LABELS:
            if re.search(r'\b' + re.escape(l.lower()) + r'\b', clean.lower()):
                return l
        return "Unknown"

    llms = []
    if RUN_GEMMA:
        llms.append({
            "name": "Gemma_2_2B",
            "adapter_dir": "/content/drive/MyDrive/NLP_Checkpoints/gemma_2_2b_it_banking_adapter/final_adapter",
            "batch_size": 32
        })
    if RUN_MISTRAL:
        llms.append({
            "name": "Mistral_7B",
            "adapter_dir": "/content/drive/MyDrive/NLP_Checkpoints/mistral_7b_banking_adapter/final_adapter",
            "batch_size": 16
        })

    for llm in llms:
        l_name = llm["name"]
        a_dir = llm["adapter_dir"]
        b_size = llm["batch_size"]

        if not os.path.exists(a_dir):
            print(f"[WARNING] Adapter directory not found: {a_dir}. Skipping.")
            continue

        print("\n" + "="*70)
        print(f"--> LOADING {l_name} FROM FINAL ADAPTER...")
        print("="*70)

        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=a_dir,
            max_seq_length=256,
            dtype=None,
            load_in_4bit=True
        )
        FastLanguageModel.for_inference(model)

        l_preds = []
        for i in tqdm(range(0, len(test_texts), b_size), desc=f"{l_name} Evaluation"):
            batch = test_texts[i : i + b_size]
            prompts = [create_few_shot_prompt(t) for t in batch]
            formatted = [tokenizer.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True) for p in prompts]
            inputs = tokenizer(formatted, return_tensors="pt", padding=True, truncation=True, max_length=256).to("cuda")
            with torch.no_grad():
                outputs = model.generate(**inputs, max_new_tokens=15, temperature=0.1, do_sample=False, use_cache=True)
            for j, out in enumerate(outputs):
                in_len = inputs["input_ids"][j].shape[0]
                gen_text = tokenizer.decode(out[in_len:], skip_special_tokens=True)
                l_preds.append(parse_pred(gen_text))

        l_acc = (np.array(l_preds) == np.array(test_labels)).mean() * 100.0
        print(f">>> [CONFIRMED] {l_name} Accuracy: {l_acc:.2f}% <<<")

        out_file = os.path.join(OUTPUT_DIR, f"{l_name.lower()}_test_predictions.json")
        with open(out_file, "w", encoding="utf-8") as f:
            json.dump({"y_true": test_labels, "y_pred": l_preds, "texts": test_texts}, f)
        print(f"--> [SAVED] {out_file}")

        del model, tokenizer
        torch.cuda.empty_cache()

print(f"\n==================================================================")
print(f"[COMPLETE] Real prediction files saved in Google Drive at:")
print(OUTPUT_DIR)
print(f"==================================================================")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--> [READY] Loaded 3,080 test queries across 77 classes.

--> EVALUATING SciBERT ON 3,080 TEST QUERIES...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Evaluating SciBERT:   0%|          | 0/97 [00:00<?, ?it/s]

>>> [CONFIRMED] SciBERT Test Accuracy: 89.74% <<<
--> [SAVED] /content/drive/MyDrive/NLP_Checkpoints/error_analysis_empirical/scibert_seed42_test_predictions.json

--> EVALUATING RoBERTa ON 3,080 TEST QUERIES...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Evaluating RoBERTa:   0%|          | 0/97 [00:00<?, ?it/s]

... (11 outputs left out)

Unsloth 2026.11.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Mistral_7B Evaluation:   0%|          | 0/193 [00:00<?, ?it/s]

Both `max_new_tokens` (=15) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=15) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=15) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=15) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

>>> [CONFIRMED] Mistral_7B Accuracy: 93.21% <<<
--> [SAVED] /content/drive/MyDrive/NLP_Checkpoints/error_analysis_empirical/mistral_7b_test_predictions.json

[COMPLETE] Real prediction files saved in Google Drive at:
/content/drive/MyDrive/NLP_Checkpoints/error_analysis_empirical
